# PyNAS

This notebook describes how to use NAS for generating and evolving neural network models.

In [2]:
%load_ext autoreload
%autoreload 2

## 🧠 PyTorch Lightning Setup Script

This script initializes a training environment for a neural network using PyTorch Lightning and a custom NAS framework.

🔍 **Explanation**

- **Dataset Loader:** Uses RawClassifierDataModule to load data with batch_size=4, num_workers=2, and no transforms.
- **Config Parsing:** Reads logs_dir_GA and seed from config.ini.
- **Environment Setup:**  
	- Enables full column display in pandas.  
	- Ensures reproducibility with pl.seed_everything().  
	- Sets matrix multiplication precision to "medium" for optimized performance.



In [4]:
import configparser
import pandas as pd

import torch
import pytorch_lightning as pl

from pynas.core.population import Population
from datasets.RawVessels.loader import RawVesselsDataModule

# Define dataset module
root_dir = "/Data_large/marine/PythonProjects/OtherProjects/lpl-PyNas/data/TASI/DataSAR_real_refined"
dm = RawVesselsDataModule(root_dir, batch_size=4, num_workers=8, transform=None, 
                            test_size=0.15, val_size=0.15, seed=42)


In [5]:
config = configparser.ConfigParser()
config.read('config.ini')
def setting():
    pd.set_option('display.max_colwidth', None)
    # Logging
    logs_directory = str(config['GA']['logs_dir_GA'])
    # Torch stuff
    seed = config.getint(section='Computation', option='seed')
    pl.seed_everything(seed=seed, workers=True)  # For reproducibility
    torch.set_float32_matmul_precision("medium")  # to make lightning happy
setting()

Seed set to 42


In [6]:
# Test the data module to ensure it works properly
print("Testing RawVesselsDataModule...")
print(f"Root directory: {dm.root_dir}")
print(f"Batch size: {dm.batch_size}")
print(f"Number of workers: {dm.num_workers}")

# Setup the data module
dm.setup()

# Get train, validation, and test dataloaders
train_loader = dm.train_dataloader()
val_loader = dm.val_dataloader()
test_loader = dm.test_dataloader()

print(f"Train dataset size: {len(train_loader.dataset)}")
print(f"Validation dataset size: {len(val_loader.dataset)}")
print(f"Test dataset size: {len(test_loader.dataset)}")

# Test loading a batch
try:
    batch = next(iter(train_loader))
    x, y = batch
    print(f"Input shape: {x.shape}")
    print(f"Target shape: {y.shape}")
    print(f"Input data type: {x.dtype}")
    print(f"Target data type: {y.dtype}")
    print("✅ Data module is working properly!")
except Exception as e:
    print(f"❌ Error loading batch: {e}")

Testing RawVesselsDataModule...
Root directory: /Data_large/marine/PythonProjects/OtherProjects/lpl-PyNas/data/TASI/DataSAR_real_refined
Batch size: 4
Number of workers: 8
Train dataset size: 1349
Validation dataset size: 289
Test dataset size: 289


terminate called without an active exception
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7fa78fc0b600>
Traceback (most recent call last):
  File "/Data_large/marine/PythonProjects/OtherProjects/lpl-PyNas/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py", line 1663, in __del__
    self._shutdown_workers()
  File "/Data_large/marine/PythonProjects/OtherProjects/lpl-PyNas/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py", line 1627, in _shutdown_workers
    w.join(timeout=_utils.MP_STATUS_CHECK_INTERVAL)
  File "/home/vessel/.local/share/pdm/python/cpython@3.11.12/lib/python3.11/multiprocessing/process.py", line 149, in join
    res = self._popen.wait(timeout)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/vessel/.local/share/pdm/python/cpython@3.11.12/lib/python3.11/multiprocessing/popen_fork.py", line 40, in wait
    if not wait([self.sentinel], timeout):
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/vessel/.

Input shape: torch.Size([4, 2, 1200, 1200])
Target shape: torch.Size([4, 1200, 1200])
Input data type: torch.float64
Target data type: torch.uint8
✅ Data module is working properly!


## ⚙️ Genetic Algorithm Model Setup

This section of code configures parameters for a Genetic Algorithm (GA)-based Neural Architecture Search (NAS) using the Population class.

In [2]:
# Model parameters
max_layers = 5 # Maximum number of layers in the model: composed of normal cell (Conv Block) and reduction cell (i.e. Pooling layer)
max_iter = int(config['GA']['max_iterations'])
# GA parameters
n_individuals = int(config['GA']['population_size'])
mating_pool_cutoff = float(config['GA']['mating_pool_cutoff'])
mutation_probability = float(config['GA']['mutation_probability'])

pop = Population(n_individuals=20, max_layers=max_layers, dm=dm, max_parameters=400_000)

### 🧬 Initial Population (`Population` Object)

The initial population defines the starting point for the Genetic Algorithm (GA)-based search. It consists of a set of randomly generated neural architectures (individuals), each encoded with:

- A **variable number of layers** (up to `max_layers`)
- **Random hyperparameters and layer types** within predefined search constraints
- A **bounded total parameter count** (`max_parameters`) to limit model complexity


In [3]:
pop.initial_poll()

Generating Population: 100%|██████████| 20/20 [00:14<00:00,  1.36it/s]


### 🔄 `train()` vs `evolve()` in a GA-based NAS Framework

In the context of the `Population` class within a Genetic Algorithm (GA)-driven Neural Architecture Search (NAS), the methods `train()` and `evolve()` serve distinct purposes in the evolutionary pipeline.

---

### 🏋️‍♂️ `train()`

Trains all individuals (i.e., neural architectures) in the current population.

#### 📌 Responsibilities:
- Performs forward and backward passes on the dataset (`dm`).
- Optimizes model weights using a standard training loop.
- Evaluates performance (fitness), typically via validation accuracy or loss.
- Stores fitness scores used for selection in the GA.

#### ✅ Outcome:
Each individual's **fitness value** is updated and can now be ranked for survival and reproduction.

---

### 🧬 `evolve()`

Applies evolutionary operations to produce the **next generation** of architectures.

#### 📌 Responsibilities:
- **Selection**: Ranks individuals by fitness and selects top performers (according to `mating_pool_cutoff`).
- **Crossover**: Combines architecture components (e.g., layer types, connections) from parent individuals.
- **Mutation**: Applies random changes (with `mutation_probability`) to maintain diversity.
- **Population Replacement**: Creates a new generation of individuals.

#### ✅ Outcome:
A **new population** is generated with architectural variations derived from the most promising candidates of the previous generation.

In [ ]:
for _ in range(max_iter):
    pop.train_generation(task='classification', lr=0.001, epochs=15, batch_size=32) 
    pop.evolve(mating_pool_cutoff=mating_pool_cutoff, mutation_probability=0.85, k_best=1, n_random=3)

### 📄 `load_dataframe()` Method – What It Does

The `load_dataframe` method in the `Population` class is used to retrieve 📦 **stored results or evaluation metrics** from the training and evolution process of the models.

When calling `pop.load_dataframe(9)`, it loads data such as:
- 📊 **Performance metrics**
- 📉 **Loss values**
- 🧠 **Architectural configurations**

These were saved during the evolutionary search.

You can use this data for:
- 🔍 **Analysis** of model performance
- 📈 **Visualization** of evolutionary dynamics
- 🛠️ **Further processing** like re-training or model selection

⚠️ **Note**: The index passed (e.g., `9`) must correspond to the specific generation or result set you want to inspect.

In [ ]:
pop.load_dataframe(9)

# Inference

Using the evaluated and saved model. We use the traced pytroch model (.pt) to load and execute inference.

In [ ]:
# Load the saved TorchScript model and test with a dummy input.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

save_path = "model_and_architecture.pt"
loaded_model = torch.jit.load(save_path, map_location=device)
loaded_model.eval()

# Ensure input is moved to the correct device
example_input = torch.randn(1, *dm.input_shape).to(device)
example_input = example_input.to(device)

with torch.no_grad():
    output = loaded_model(example_input)
print("Output from the loaded model:", output)